In [34]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
from mabwiser.mab import MAB, LearningPolicy
import random
import pandas as pd

In [36]:
arms = ["Cards", "Traditional", "Swipe"] # Various UX
jobseeker = pd.DataFrame([{"id": 1, "age": 1}, {"id": 2, "age": 2}, {"id": 3, "age": 3}]) # Example job seeker datas
company = pd.DataFrame([{"id": 1, "difficulty": 1}, {"id": 2, "difficulty": 1}]) # Example company datas
jobs = pd.DataFrame([{"company_id": 1, "id": 1, "difficulty": 1}, {"company_id": 2, "id": 2, "difficulty": 2}]) # Example job datas
N_RUNS = 1000 # How many runs?
# TODO: Please adjust the decisions and rewards accordingly.
# TODO: ...or you may as well integrate into the driver.

In [37]:
linUCB = MAB(arms, LearningPolicy.LinUCB(alpha=1.25)) # Linear UCB, for contextual.
linTS = MAB(arms, LearningPolicy.LinTS(alpha=1.25)) # Linear Thompson Sampling, for contextual.

linUCB.fit([arms[0]], [0], [[0, 0, 0]]) # Initialize with dummy data for contextual learning
linTS.fit([arms[0]], [0], [[0, 0, 0]]) # Initialize with dummy data for contextual learning
# TODO: According to your needs, do whatever adjustment you like.
# Take note that this is ONLINE LEARNING!

In [38]:
def fit_score(jobseeker_context, company_context, job_context):
    # Example fit score, you can adjust this.
    return jobseeker_context["age"] * company_context["difficulty"] * job_context["difficulty"]

In [39]:
for e in range(N_RUNS):
    jobseeker_context = jobseeker.sample(1).iloc[0].to_dict()
    company_context = company.sample(1).iloc[0].to_dict()
    job_context = jobs[jobs['company_id'] == company_context['id']].sample(1).iloc[0].to_dict()
    context = [[jobseeker_context['age'], company_context['difficulty'], job_context['difficulty']]]
    arm = linUCB.predict(context)[0]  # Use LinUCB to select arm based on context
    reward_score = fit_score(jobseeker_context, company_context, job_context)  # Calculate fit score
    reward = reward_score  # Assuming reward_score is the reward for the MAB
    linUCB.partial_fit([arm], [reward], context)  # Update the MAB with the new data
    print(f"Job Seeker Context: {jobseeker_context}, Company Context: {company_context}, Job Context: {job_context}")
    print(f"Selected Arm: {arm}, Fit Score: {fit_score}, Reward: {reward}")
    print("MAB updated")
print("End of simulation.")

Job Seeker Context: {'id': 2, 'age': 2}, Company Context: {'id': 2, 'difficulty': 1}, Job Context: {'company_id': 2, 'id': 2, 'difficulty': 2}
Selected Arm: C, Fit Score: <function fit_score at 0x000001B889738D50>, Reward: 4
MAB updated
Job Seeker Context: {'id': 2, 'age': 2}, Company Context: {'id': 1, 'difficulty': 1}, Job Context: {'company_id': 1, 'id': 1, 'difficulty': 1}
Selected Arm: C, Fit Score: <function fit_score at 0x000001B889738D50>, Reward: 2
MAB updated
Job Seeker Context: {'id': 1, 'age': 1}, Company Context: {'id': 2, 'difficulty': 1}, Job Context: {'company_id': 2, 'id': 2, 'difficulty': 2}
Selected Arm: C, Fit Score: <function fit_score at 0x000001B889738D50>, Reward: 2
MAB updated
Job Seeker Context: {'id': 2, 'age': 2}, Company Context: {'id': 1, 'difficulty': 1}, Job Context: {'company_id': 1, 'id': 1, 'difficulty': 1}
Selected Arm: C, Fit Score: <function fit_score at 0x000001B889738D50>, Reward: 2
MAB updated
Job Seeker Context: {'id': 2, 'age': 2}, Company Con